# Síntesis Global de Modelos y Arquitectura de Despliegue Web

## 1. Cuadro Maestro Comparativo: Regresión (Predicción de Producción en qq)
Evaluación sobre el dataset curado `0901_Produccion_Arroz_V2_Regresion.xlsx` (1,373 registros, sin outliers extremos > 3,000 Ha, sin fuga de datos):

| Rango | Algoritmo | Script Fuente | $R^2$ Score | MAE (qq) | RMSE (qq) | MAPE (%) | Veredicto de Producción |
|:---:|---|---|:---:|:---:|:---:|:---:|---|
| **1°** | **LightGBM V2** | `13_Boosting_Regresion_V2.py` | **0.7682** | **1,254.62** | **4,467.49** | **45.20%** | **Ganador Absoluto (Backend API)** |
| **2°** | **Regresión Lineal V3** | `10_Regresion_Lineal_V3.py` | 0.6787 | 1,470.68 | 5,259.79 | 49.03% | **Ganador Absoluto (Frontend Serverless)** |
| **3°** | **Árbol de Decisión V2** | `12_Arbol_Decision_Regresion_V2.py` | 0.6704 | 1,435.64 | 5,327.80 | 49.56% | Interpretable pero escalonado |
| **4°** | **Polinomial Ridge V3** | `11_Regresion_Polinomial_V3.py` | 0.5651 | 1,365.46 | 6,119.86 | 51.05% | Regularizado ($\alpha=10$), menor parsimonia |
| **5°** | **CatBoost V2** | `14_GradientBoosting_Regresion_V2.py` | 0.3582 | 1,506.54 | 7,434.33 | 51.71% | Robusto regionalmente, mayor varianza residual |
| **6°** | **XGBoost V2** | `14_GradientBoosting_Regresion_V2.py` | 0.4545 | 1,426.35 | 6,854.22 | 52.64% | Ajuste intermedio, penalizado por One-Hot |
| **7°** | **AdaBoost V2** | `13_Boosting_Regresion_V2.py` | 0.4140 | 1,777.09 | 7,103.87 | 49.78% | Penalización excesiva en colas |
| **8°** | **SVR V2** | `15_SVM_Regresion_V2.py` | 0.2207 | 1,485.89 | 8,192.08 | 55.58% | Recuperado, no competitivo en exactitud |

## 2. Cuadro Maestro Comparativo: Clasificación (Detección de Invierno)
Evaluación sobre el dataset balanceado `0901_Produccion_Arroz_V3_Clasificacion.xlsx` (280 registros, 140 Verano vs 140 Invierno, partición test 42 vs 42):

| Rango | Algoritmo | Script Fuente | Accuracy | Recall (Inv) | Precision (Inv) | F1-Score | ROC-AUC | Veredicto de Producción |
|:---:|---|---|:---:|:---:|:---:|:---:|:---:|---|
| **1°** | **CatBoost V2** | `15_GradientBoosting_Clasificacion_V2.py` | **79.76%** | **85.71% (36/42)** | 76.60% | **0.8090** | **0.8679** | **Ganador Absoluto (Producción)** |
| **2°** | **XGBoost V2** | `15_GradientBoosting_Clasificacion_V2.py` | 78.57% | 80.95% (34/42) | 77.27% | 0.7907 | 0.8583 | Alto desempeño, mayor consumo RAM |
| **3°** | **Árbol de Decisión V2** | `13_Arbol_Decision_Clasificacion_V2.py` | 69.05% | 64.29% (27/42) | 71.05% | 0.6750 | 0.7364 | Mejor modelo de caja blanca |
| **4°** | **LightGBM V2** | `14_Boosting_Clasificacion_V2.py` | 65.48% | 61.90% (26/42) | 66.67% | 0.6420 | 0.7183 | Equilibrado y compacto |
| **5°** | **Naive Bayes Gaussiano V2** | `12_Naive_Bayes_Clasificacion_V2.py` | 67.86% | 54.76% (23/42) | 74.19% | 0.6301 | 0.7120 | Rápido, ideal para microservicios |
| **6°** | **Regresión Logística V3** | `10_Regresion_Logistica_V3.py` | 67.86% | 47.62% (20/42) | **80.00%** | 0.5970 | 0.7426 | Máxima precisión, interpretable |
| **7°** | **KNN V3** | `11_KNN_V3.py` | 60.71% | 57.14% (24/42) | 61.54% | 0.5926 | 0.6842 | Requiere conjunto en memoria |
| **8°** | **AdaBoost V2** | `14_Boosting_Clasificacion_V2.py` | 65.48% | 45.24% (19/42) | 76.00% | 0.5672 | 0.6610 | Sensibilidad moderada |
| **9°** | **SVC (RBF) V2** | `16_SVM_Clasificacion_V2.py` | 59.52% | 30.95% (13/42) | 72.22% | 0.4333 | 0.7574 | Demasiado conservador en frontera |

## 3. Arquitectura y Veredicto para Despliegue Web

### Selección Definitiva:
1. **Estimador Numérico de Cosecha:**
   * **Opción Principal (Backend Python / API FastAPI):** `LightGBM Regresión V2` (archivo serializado `.joblib`), alcanzando un MAPE de 45.20% y MAE de 1,254.62 qq.
   * **Opción Alternativa (Serverless Frontend en JavaScript):** `Regresión Lineal V3`, implementable en el navegador mediante:
     ```javascript
     const logSuperficie = Math.log1p(superficieHa);
     const logPrediccion = 3.5059 + 0.9667 * logSuperficie;
     const produccionQuintales = Math.expm1(logPrediccion);
     ```
2. **Clasificador Estacional:**
   * **Elección Única:** `CatBoost Clasificador V2`, capaz de identificar el 85.71% de los cultivos invernales con un ROC-AUC de 0.8679 y procesamiento nativo de variables geográficas en texto plano.